In [1]:
from pathlib import Path
from collections import Counter
import os
import random
import re

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

In [2]:
ROOT = Path.cwd().parent

DATA_PATH = ROOT / "dataset" / "data" / "combined_train.csv"
GLOVE_PATH = ROOT / "dataset" / "embeddings" / "glove.6B.100d.txt"
CHECKPOINT_DIR = ROOT / "checkpoints"
MODEL_PATH = CHECKPOINT_DIR / "atae_lstm_best.pt"
HISTORY_PATH = CHECKPOINT_DIR / "atae_lstm_training_history.csv"

CHECKPOINT_DIR.mkdir(exist_ok=True)

SEED = 42

EMBEDDING_DIM = 100
HIDDEN_DIM = 128
BATCH_SIZE = 32
EPOCHS = 15
LEARNING_RATE = 0.001
DROPOUT = 0.30
WEIGHT_DECAY = 1e-5

MAX_SENTENCE_LEN = 100
MAX_ASPECT_LEN = 10
MIN_WORD_FREQUENCY = 2

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
PAD_ID = 0
UNK_ID = 1

LABEL_TO_ID = {
    "negative": 0,
    "neutral": 1,
    "positive": 2,
    "conflict": 3
}

ID_TO_LABEL = {
    label_id: label
    for label, label_id in LABEL_TO_ID.items()
}

print("Project root:", ROOT)
print("Training data:", DATA_PATH)
print("GloVe file:", GLOVE_PATH)
print("Model output:", MODEL_PATH)

Project root: c:\New folder\Projects\sentiment analysis
Training data: c:\New folder\Projects\sentiment analysis\dataset\data\combined_train.csv
GloVe file: c:\New folder\Projects\sentiment analysis\dataset\embeddings\glove.6B.100d.txt
Model output: c:\New folder\Projects\sentiment analysis\checkpoints\atae_lstm_best.pt


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print(
        "Allocated GPU memory:",
        round(torch.cuda.memory_allocated() / 1024**2, 2),
        "MB"
    )
else:
    print("CUDA unavailable. Training will run on CPU.")

Device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
CUDA version: 12.8
Allocated GPU memory: 0.0 MB


In [4]:
def tokenize(text):
    """
    Keeps normal words, numbers, and contractions.
    Examples:
    "8GB RAM isn't enough!" -> ["8gb", "ram", "isn't", "enough"]
    """
    text = str(text).lower()
    return re.findall(r"[a-z0-9]+(?:'[a-z]+)?", text)


LABEL_FIXES = {
    "postive": "positive",
    "po": "positive",
}

VALID_LABELS = {
    "negative",
    "neutral",
    "positive",
    "conflict",
}


def clean_absa_data(df):
    df = df.copy()

    df["polarity"] = (
        df["polarity"]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace(LABEL_FIXES)
    )

    df = df.dropna(
        subset=["text", "aspect_term", "polarity"]
    )

    df = df[
        df["text"].astype(str).str.strip().ne("") &
        df["aspect_term"].astype(str).str.strip().ne("")
    ].copy()

    df = df[
        df["polarity"].isin(VALID_LABELS)
    ].copy()

    df["text_tokens"] = df["text"].apply(tokenize)
    df["aspect_tokens"] = df["aspect_term"].apply(tokenize)
    df["label"] = df["polarity"].map(LABEL_TO_ID)

    df = df[
        df["text_tokens"].map(len).gt(0) &
        df["aspect_tokens"].map(len).gt(0)
    ].copy()

    return df.reset_index(drop=True)

In [5]:
data = pd.read_csv(DATA_PATH)

print("Original shape:", data.shape)

print("\nOriginal polarity values:")
print(data["polarity"].value_counts(dropna=False))

data = clean_absa_data(data)

print("\nCleaned shape:", data.shape)

print("\nCleaned polarity distribution:")
print(data["polarity"].value_counts())

display(
    data[
        ["text", "aspect_term", "polarity", "text_tokens", "aspect_tokens"]
    ].head()
)

Original shape: (2734, 5)

Original polarity values:
polarity
positive    1196
negative    1018
neutral      473
conflict      45
NaN            2
Name: count, dtype: int64

Cleaned shape: (2721, 8)

Cleaned polarity distribution:
polarity
positive    1193
negative    1013
neutral      470
conflict      45
Name: count, dtype: int64


,text,aspect_term,polarity,text_tokens,aspect_tokens
0,lenovo personally maybe,lenovo,positive,"[lenovo, personally, maybe]",[lenovo]
1,legion like intel chipset great laptop value b...,intel,positive,"[legion, like, intel, chipset, great, laptop, ...",[intel]
2,legion like intel chipset great laptop value b...,battery life,positive,"[legion, like, intel, chipset, great, laptop, ...","[battery, life]"
3,definitely consider lenovo gaming laptop,lenovo,positive,"[definitely, consider, lenovo, gaming, laptop]",[lenovo]
4,hp user not hp,hp,negative,"[hp, user, not, hp]",[hp]


In [6]:
train_df, temp_df = train_test_split(
    data,
    test_size=0.20,
    random_state=SEED,
    stratify=data["label"]
)

validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

print("\nTrain labels:")
print(train_df["polarity"].value_counts())

print("\nValidation labels:")
print(validation_df["polarity"].value_counts())

print("\nTest labels:")
print(test_df["polarity"].value_counts())

Train: (2176, 8)
Validation: (272, 8)
Test: (273, 8)

Train labels:
polarity
positive    954
negative    810
neutral     376
conflict     36
Name: count, dtype: int64

Validation labels:
polarity
positive    119
negative    101
neutral      47
conflict      5
Name: count, dtype: int64

Test labels:
polarity
positive    120
negative    102
neutral      47
conflict      4
Name: count, dtype: int64


In [7]:
word_counter = Counter()

for tokens in train_df["text_tokens"]:
    word_counter.update(tokens)

for tokens in train_df["aspect_tokens"]:
    word_counter.update(tokens)

vocab = {
    PAD_TOKEN: PAD_ID,
    UNK_TOKEN: UNK_ID
}

for word, count in word_counter.items():
    if count >= MIN_WORD_FREQUENCY:
        vocab[word] = len(vocab)

id_to_word = {
    word_id: word
    for word, word_id in vocab.items()
}

print("Vocabulary size:", len(vocab))
print("First 30 vocabulary entries:")
print(list(vocab.items())[:30])

Vocabulary size: 1883
First 30 vocabulary entries:
[('<pad>', 0), ('<unk>', 1), ('love', 2), ('particular', 3), ('mac', 4), ('fast', 5), ('great', 6), ('size', 7), ('fantastic', 8), ('feature', 9), ('like', 10), ('lighted', 11), ('keyboard', 12), ('easy', 13), ('mouse', 14), ('pad', 15), ('battery', 16), ('completely', 17), ('gb', 18), ('ram', 19), ('rough', 20), ('want', 21), ('gaming', 22), ('plain', 23), ('simple', 24), ('run', 25), ('load', 26), ('pretty', 27), ('poor', 28), ('laptop', 29)]


In [8]:
if not GLOVE_PATH.exists():
    raise FileNotFoundError(
        f"GloVe file not found at:\n{GLOVE_PATH}\n\n"
        "Download glove.6B.zip from Stanford GloVe, extract it, "
        "and place glove.6B.100d.txt inside dataset/embeddings/."
    )

def load_glove_embeddings(
    glove_path,
    vocabulary,
    embedding_dim
):
    """
    Create an embedding matrix aligned with this project's vocabulary.

    - PAD token gets all zeros.
    - Words present in GloVe get pretrained vectors.
    - Unknown vocabulary words get small random vectors.
    """
    embedding_matrix = np.random.normal(
        loc=0.0,
        scale=0.05,
        size=(len(vocabulary), embedding_dim)
    ).astype(np.float32)

    embedding_matrix[PAD_ID] = np.zeros(
        embedding_dim,
        dtype=np.float32
    )

    found = 0

    with open(glove_path, "r", encoding="utf-8") as glove_file:
        for line in glove_file:
            values = line.rstrip().split(" ")

            word = values[0]

            if word not in vocabulary:
                continue

            vector = np.asarray(
                values[1:],
                dtype=np.float32
            )

            if len(vector) != embedding_dim:
                continue

            embedding_matrix[vocabulary[word]] = vector
            found += 1

    total_real_words = max(len(vocabulary) - 2, 1)
    coverage = found / total_real_words * 100

    print(f"Vocabulary words found in GloVe: {found:,}")
    print(f"GloVe coverage: {coverage:.2f}%")

    return torch.tensor(
        embedding_matrix,
        dtype=torch.float32
    )

In [9]:
embedding_matrix = load_glove_embeddings(
    glove_path=GLOVE_PATH,
    vocabulary=vocab,
    embedding_dim=EMBEDDING_DIM
)

print("Embedding matrix shape:", embedding_matrix.shape)

Vocabulary words found in GloVe: 1,796
GloVe coverage: 95.48%
Embedding matrix shape: torch.Size([1883, 100])


In [10]:
def encode_tokens(tokens, vocabulary, max_length):
    token_ids = [
        vocabulary.get(token, UNK_ID)
        for token in tokens[:max_length]
    ]

    padding_needed = max_length - len(token_ids)

    if padding_needed > 0:
        token_ids.extend([PAD_ID] * padding_needed)

    return token_ids

In [11]:
class ABSADataset(Dataset):
    def __init__(self, dataframe, vocabulary):
        self.dataframe = dataframe.reset_index(drop=True)
        self.vocabulary = vocabulary

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        sentence_ids = encode_tokens(
            tokens=row["text_tokens"],
            vocabulary=self.vocabulary,
            max_length=MAX_SENTENCE_LEN
        )

        aspect_ids = encode_tokens(
            tokens=row["aspect_tokens"],
            vocabulary=self.vocabulary,
            max_length=MAX_ASPECT_LEN
        )

        return {
            "sentence_ids": torch.tensor(
                sentence_ids,
                dtype=torch.long
            ),
            "aspect_ids": torch.tensor(
                aspect_ids,
                dtype=torch.long
            ),
            "label": torch.tensor(
                row["label"],
                dtype=torch.long
            )
        }

In [12]:
train_dataset = ABSADataset(train_df, vocab)
validation_dataset = ABSADataset(validation_df, vocab)
test_dataset = ABSADataset(test_df, vocab)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

sample_batch = next(iter(train_loader))

print("Sentence batch shape:", sample_batch["sentence_ids"].shape)
print("Aspect batch shape:", sample_batch["aspect_ids"].shape)
print("Label batch shape:", sample_batch["label"].shape)

Sentence batch shape: torch.Size([32, 100])
Aspect batch shape: torch.Size([32, 10])
Label batch shape: torch.Size([32])


In [13]:
class ATAELSTM(nn.Module):
    """
    Aspect-aware attention-based BiLSTM for aspect polarity classification.

    Input:
    - sentence_ids: [batch_size, max_sentence_len]
    - aspect_ids:  [batch_size, max_aspect_len]

    Architecture:
    1. Word embeddings initialized from pretrained GloVe.
    2. Mean-pool multi-word aspect embeddings.
    3. Concatenate each sentence word embedding with aspect vector.
    4. Encode with a Bidirectional LSTM.
    5. Calculate aspect-aware attention over LSTM states.
    6. Predict one of four polarity classes through softmax logits.
    """

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_classes,
        embedding_weights,
        dropout=0.30
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=PAD_ID
        )

        self.embedding.weight.data.copy_(
            embedding_weights
        )

        # Fine-tune GloVe embeddings during task-specific training.
        self.embedding.weight.requires_grad = True

        self.lstm = nn.LSTM(
            input_size=embedding_dim * 2,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        lstm_output_dim = hidden_dim * 2

        # Attention receives:
        # BiLSTM hidden state + corresponding aspect vector.
        self.attention_projection = nn.Linear(
            lstm_output_dim + embedding_dim,
            lstm_output_dim
        )

        self.attention_score = nn.Linear(
            lstm_output_dim,
            1,
            bias=False
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            lstm_output_dim,
            num_classes
        )

    def forward(self, sentence_ids, aspect_ids):
        # True where a sentence token is not padding.
        sentence_mask = sentence_ids.ne(PAD_ID)

        # [batch, sentence_length, embedding_dim]
        sentence_embeddings = self.embedding(sentence_ids)

        # [batch, aspect_length, embedding_dim]
        aspect_embeddings = self.embedding(aspect_ids)

        # Average embeddings for multi-word aspect terms.
        aspect_mask = aspect_ids.ne(PAD_ID).unsqueeze(-1)

        aspect_sum = (
            aspect_embeddings * aspect_mask
        ).sum(dim=1)

        aspect_length = aspect_mask.sum(dim=1).clamp(min=1)

        # [batch, embedding_dim]
        aspect_vector = aspect_sum / aspect_length

        # Copy aspect vector across every word position.
        expanded_aspect = aspect_vector.unsqueeze(1).expand(
            -1,
            sentence_embeddings.size(1),
            -1
        )

        # Feed [word_embedding ; aspect_embedding] into BiLSTM.
        lstm_input = torch.cat(
            [sentence_embeddings, expanded_aspect],
            dim=-1
        )

        # [batch, sentence_length, 2 * hidden_dim]
        lstm_output, _ = self.lstm(lstm_input)

        # Calculate aspect-aware attention scores.
        attention_input = torch.cat(
            [lstm_output, expanded_aspect],
            dim=-1
        )

        attention_hidden = torch.tanh(
            self.attention_projection(attention_input)
        )

        attention_scores = self.attention_score(
            attention_hidden
        ).squeeze(-1)

        # Never assign attention to padded sentence positions.
        attention_scores = attention_scores.masked_fill(
            ~sentence_mask,
            -1e9
        )

        attention_weights = torch.softmax(
            attention_scores,
            dim=1
        )

        # Weighted sum of BiLSTM states.
        sentence_representation = torch.bmm(
            attention_weights.unsqueeze(1),
            lstm_output
        ).squeeze(1)

        sentence_representation = self.dropout(
            sentence_representation
        )

        # Raw class scores. CrossEntropyLoss applies softmax internally.
        logits = self.classifier(sentence_representation)

        return logits, attention_weights

In [14]:
model = ATAELSTM(
    vocab_size=len(vocab),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=len(LABEL_TO_ID),
    embedding_weights=embedding_matrix,
    dropout=DROPOUT
).to(device)

print(model)

ATAELSTM(
  (embedding): Embedding(1883, 100, padding_idx=0)
  (lstm): LSTM(200, 128, batch_first=True, bidirectional=True)
  (attention_projection): Linear(in_features=356, out_features=256, bias=True)
  (attention_score): Linear(in_features=256, out_features=1, bias=False)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=256, out_features=4, bias=True)
)


In [15]:
train_class_counts = (
    train_df["label"]
    .value_counts()
    .sort_index()
)

class_weights = []

for class_id in range(len(LABEL_TO_ID)):
    class_count = train_class_counts.get(class_id, 1)

    weight = len(train_df) / (
        len(LABEL_TO_ID) * class_count
    )

    class_weights.append(weight)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

print("Class weights:")

for class_id, weight in enumerate(
    class_weights.detach().cpu().numpy()
):
    print(
        f"{ID_TO_LABEL[class_id]}: "
        f"{weight:.4f}"
    )

Class weights:
negative: 0.6716
neutral: 1.4468
positive: 0.5702
conflict: 15.1111


In [16]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

In [17]:
def run_epoch(model, data_loader, optimizer=None):
    is_training = optimizer is not None

    if is_training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_labels = []
    all_predictions = []

    for batch in data_loader:
        sentence_ids = batch["sentence_ids"].to(device)
        aspect_ids = batch["aspect_ids"].to(device)
        labels = batch["label"].to(device)

        if is_training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_training):
            logits, _ = model(
                sentence_ids,
                aspect_ids
            )

            loss = criterion(logits, labels)

            if is_training:
                loss.backward()

                # Stops an unstable gradient update.
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=5.0
                )

                optimizer.step()

        total_loss += loss.item() * labels.size(0)

        predictions = torch.argmax(
            logits,
            dim=1
        )

        all_labels.extend(
            labels.detach().cpu().numpy()
        )

        all_predictions.extend(
            predictions.detach().cpu().numpy()
        )

    average_loss = total_loss / len(data_loader.dataset)

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    return {
        "loss": average_loss,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "labels": all_labels,
        "predictions": all_predictions
    }

In [18]:
best_validation_f1 = -1.0
history = []

for epoch in range(1, EPOCHS + 1):
    training_result = run_epoch(
        model=model,
        data_loader=train_loader,
        optimizer=optimizer
    )

    validation_result = run_epoch(
        model=model,
        data_loader=validation_loader
    )

    scheduler.step(validation_result["macro_f1"])

    current_lr = optimizer.param_groups[0]["lr"]

    row = {
        "epoch": epoch,
        "train_loss": training_result["loss"],
        "train_accuracy": training_result["accuracy"],
        "train_macro_f1": training_result["macro_f1"],
        "validation_loss": validation_result["loss"],
        "validation_accuracy": validation_result["accuracy"],
        "validation_macro_f1": validation_result["macro_f1"],
        "learning_rate": current_lr
    }

    history.append(row)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {training_result['loss']:.4f} | "
        f"Train F1: {training_result['macro_f1']:.4f} | "
        f"Val Loss: {validation_result['loss']:.4f} | "
        f"Val F1: {validation_result['macro_f1']:.4f} | "
        f"LR: {current_lr:.6f}"
    )

    if validation_result["macro_f1"] > best_validation_f1:
        best_validation_f1 = validation_result["macro_f1"]

        checkpoint = {
            "model_state_dict": model.state_dict(),
            "vocab": vocab,
            "label_to_id": LABEL_TO_ID,
            "config": {
                "embedding_dim": EMBEDDING_DIM,
                "hidden_dim": HIDDEN_DIM,
                "dropout": DROPOUT,
                "max_sentence_len": MAX_SENTENCE_LEN,
                "max_aspect_len": MAX_ASPECT_LEN
            },
            "best_validation_macro_f1": best_validation_f1
        }

        torch.save(checkpoint, MODEL_PATH)

        print(
            "  ✓ Saved best model | "
            f"Validation Macro-F1: {best_validation_f1:.4f}"
        )

Epoch 01/15 | Train Loss: 1.3239 | Train F1: 0.3409 | Val Loss: 1.2365 | Val F1: 0.4500 | LR: 0.001000
  ✓ Saved best model | Validation Macro-F1: 0.4500
Epoch 02/15 | Train Loss: 1.0866 | Train F1: 0.4802 | Val Loss: 1.2009 | Val F1: 0.4274 | LR: 0.001000
Epoch 03/15 | Train Loss: 0.9017 | Train F1: 0.5442 | Val Loss: 1.2136 | Val F1: 0.4219 | LR: 0.001000
Epoch 04/15 | Train Loss: 0.7390 | Train F1: 0.5905 | Val Loss: 1.3603 | Val F1: 0.4440 | LR: 0.000500
Epoch 05/15 | Train Loss: 0.5721 | Train F1: 0.6552 | Val Loss: 1.4189 | Val F1: 0.4809 | LR: 0.000500
  ✓ Saved best model | Validation Macro-F1: 0.4809
Epoch 06/15 | Train Loss: 0.4654 | Train F1: 0.7213 | Val Loss: 1.5113 | Val F1: 0.4842 | LR: 0.000500
  ✓ Saved best model | Validation Macro-F1: 0.4842
Epoch 07/15 | Train Loss: 0.4071 | Train F1: 0.7638 | Val Loss: 1.6761 | Val F1: 0.4678 | LR: 0.000500
Epoch 08/15 | Train Loss: 0.3461 | Train F1: 0.7985 | Val Loss: 1.7776 | Val F1: 0.4514 | LR: 0.000500
Epoch 09/15 | Train Los

In [19]:
history_df = pd.DataFrame(history)

history_df.to_csv(
    HISTORY_PATH,
    index=False
)

display(history_df)

print("Training history saved to:")
print(HISTORY_PATH)

,epoch,train_loss,train_accuracy,train_macro_f1,validation_loss,validation_accuracy,validation_macro_f1,learning_rate
0,1,1.323860,0.448989,0.340905,1.236478,0.540441,0.449999,0.001000
1,2,1.086626,0.609375,0.480201,1.200853,0.555147,0.427359,0.001000
2,3,0.901689,0.655331,0.544170,1.213566,0.507353,0.421881,0.001000
3,4,0.738977,0.705882,0.590532,1.360290,0.555147,0.444028,0.000500
4,5,0.572120,0.764706,0.655182,1.418871,0.613971,0.480876,0.000500
5,6,0.465358,0.811581,0.721337,1.511263,0.625000,0.484185,0.000500
6,7,0.407094,0.833640,0.763753,1.676069,0.647059,0.467836,0.000500
7,8,0.346089,0.865349,0.798537,1.777557,0.625000,0.451375,0.000500
8,9,0.292620,0.880515,0.829431,1.796580,0.613971,0.443502,0.000250
9,10,0.240602,0.906250,0.864712,1.971411,0.669118,0.479900,0.000250


Training history saved to:
c:\New folder\Projects\sentiment analysis\checkpoints\atae_lstm_training_history.csv


In [20]:
checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

print(
    "Best validation Macro-F1:",
    round(
        checkpoint["best_validation_macro_f1"],
        4
    )
)

Best validation Macro-F1: 0.4842


In [ ]:
best_model = ATAELSTM(
    vocab_size=len(checkpoint["vocab"]),
    embedding_dim=checkpoint["config"]["embedding_dim"],
    hidden_dim=checkpoint["config"]["hidden_dim"],
    num_classes=len(checkpoint["label_to_id"]),
    embedding_weights=embedding_matrix,
    dropout=checkpoint["config"]["dropout"]
).to(device)

best_model.load_state_dict(
    checkpoint["model_state_dict"]
)

test_result = run_epoch(
    model=best_model,
    data_loader=test_loader
)

print("=" * 70)
print("FINAL ATAE-LSTM TEST RESULTS")
print("=" * 70)

print("Test loss:", round(test_result["loss"], 4))ordered_label_ids = list(range(len(LABEL_TO_ID)))

ordered_label_names = [
    ID_TO_LABEL[label_id]
    for label_id in ordered_label_ids
]

print(
    classification_report(
        test_result["labels"],
        test_result["predictions"],
        labels=ordered_label_ids,
        target_names=ordered_label_names,
        zero_division=0
    )
)

print("Confusion matrix:")

print(
    confusion_matrix(
        test_result["labels"],
        test_result["predictions"],
        labels=ordered_label_ids
    )
)
print("Test accuracy:", round(test_result["accuracy"], 4))
print("Test Macro-F1:", round(test_result["macro_f1"], 4))

FINAL ATAE-LSTM TEST RESULTS
Test loss: 1.2763
Test accuracy: 0.674
Test Macro-F1: 0.54


In [22]:
ordered_label_ids = list(range(len(LABEL_TO_ID)))

ordered_label_names = [
    ID_TO_LABEL[label_id]
    for label_id in ordered_label_ids
]

print(
    classification_report(
        test_result["labels"],
        test_result["predictions"],
        labels=ordered_label_ids,
        target_names=ordered_label_names,
        zero_division=0
    )
)

print("Confusion matrix:")

print(
    confusion_matrix(
        test_result["labels"],
        test_result["predictions"],
        labels=ordered_label_ids
    )
)

              precision    recall  f1-score   support

    negative       0.67      0.73      0.70       102
     neutral       0.50      0.47      0.48        47
    positive       0.83      0.72      0.77       120
    conflict       0.13      0.50      0.21         4

    accuracy                           0.67       273
   macro avg       0.53      0.60      0.54       273
weighted avg       0.70      0.67      0.68       273

Confusion matrix:
[[74 10 11  7]
 [17 22  7  1]
 [18 11 86  5]
 [ 1  1  0  2]]


In [23]:
def predict_atae_polarity(
    text,
    aspect_term,
    model,
    vocabulary
):
    model.eval()

    text_tokens = tokenize(text)
    aspect_tokens = tokenize(aspect_term)

    sentence_ids = encode_tokens(
        tokens=text_tokens,
        vocabulary=vocabulary,
        max_length=MAX_SENTENCE_LEN
    )

    aspect_ids = encode_tokens(
        tokens=aspect_tokens,
        vocabulary=vocabulary,
        max_length=MAX_ASPECT_LEN
    )

    sentence_tensor = torch.tensor(
        [sentence_ids],
        dtype=torch.long
    ).to(device)

    aspect_tensor = torch.tensor(
        [aspect_ids],
        dtype=torch.long
    ).to(device)

    with torch.no_grad():
        logits, attention_weights = model(
            sentence_tensor,
            aspect_tensor
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )[0]

        predicted_label_id = torch.argmax(
            probabilities
        ).item()

    return {
        "text": text,
        "aspect_term": aspect_term,
        "predicted_polarity": ID_TO_LABEL[
            predicted_label_id
        ],
        "confidence": round(
            float(
                probabilities[predicted_label_id].item()
            ),
            4
        ),
        "all_probabilities": {
            ID_TO_LABEL[label_id]: round(
                float(probabilities[label_id].item()),
                4
            )
            for label_id in range(len(LABEL_TO_ID))
        },
        "attention_weights": attention_weights[
            0
        ].detach().cpu().numpy()
    }

In [24]:
sample_comment = """
The laptop is excellent for school, web browsing, coding,
and Microsoft Office. However, 8GB RAM is restrictive.
Older games may run, but AAA gaming is unrealistic.
"""

sample_aspects = [
    "school",
    "productivity",
    "8gb ram",
    "light gaming",
    "gaming performance"
]

for aspect in sample_aspects:
    result = predict_atae_polarity(
        text=sample_comment,
        aspect_term=aspect,
        model=best_model,
        vocabulary=vocab
    )

    print(
        f"{result['aspect_term']:22} → "
        f"{result['predicted_polarity']:10} | "
        f"confidence={result['confidence']}"
    )

school                 → positive   | confidence=0.8118
productivity           → positive   | confidence=0.5702
8gb ram                → negative   | confidence=0.7187
light gaming           → positive   | confidence=0.6096
gaming performance     → negative   | confidence=0.8104
